In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor
from PIL import Image
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import defaultdict
import segmentation_models_pytorch as smp
import random
import os

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# ===== 하이퍼파라미터 =====
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
batch_size = 8
img_size = 512
heatmap_size = 128
lr = 2e-4
num_epochs = 1000
num_genes = 15

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# ===== 경로 설정 =====
data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/'
create_dir(model_path)

print(f'Device: {device}')
print(f'Batch size: {batch_size}')
print(f'Learning rate: {lr}')
print(f'Image size: {img_size}x{img_size}')
print(f'Heatmap size: {heatmap_size}x{heatmap_size}')
print(f'Num genes: {num_genes}')

In [ ]:
# ===== Dataset 클래스 =====
class CustomDataset(Dataset):
    def __init__(self, image_list, label_list, augment=False):
        self.img_paths = image_list
        self.label_paths = label_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def trans(self, image, label):
        if random.random() > 0.5:
            image = transforms.functional.hflip(image)
            label = torch.flip(label, dims=[2])
        if random.random() > 0.5:
            image = transforms.functional.vflip(image)
            label = torch.flip(label, dims=[1])
        return image, label

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = Image.open(self.img_paths[idx]).convert('RGB')
        image = self.to_tensor(image)  # (3, 512, 512)

        label = np.load(self.label_paths[idx])  # (15, 128, 128)
        label = torch.from_numpy(label).float()

        if self.augment:
            image, label = self.trans(image, label)

        return image, label

# ===== 데이터 수집 =====
img_list = sorted(glob(f'{data_dir}/image/**/*.tiff', recursive=True))
label_list = [p.replace('/image/', '/label/').replace('.tiff', '.npy') for p in img_list]

# 라벨 파일 존재 확인
valid_pairs = [(img, lbl) for img, lbl in zip(img_list, label_list) if os.path.exists(lbl)]
img_list = [p[0] for p in valid_pairs]
label_list = [p[1] for p in valid_pairs]

print(f'Total valid patches: {len(img_list)}')

# ===== Train/Val 분할 =====
train_imgs, val_imgs, train_labels, val_labels = train_test_split(
    img_list, label_list, test_size=0.2, random_state=42
)

print(f'Train: {len(train_imgs)}, Val: {len(val_imgs)}')

# ===== DataLoader 생성 =====
train_dataset = CustomDataset(train_imgs, train_labels, augment=True)
val_dataset = CustomDataset(val_imgs, val_labels, augment=False)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=4, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}')

In [ ]:
# ===== 모델 정의 =====
class GeneExpressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=15):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights='imagenet',
            in_channels=3,
            classes=num_genes,
        )

    def forward(self, x):
        out = self.unet(x)  # (B, 15, 512, 512)
        out = F.interpolate(out, size=(heatmap_size, heatmap_size), mode='bilinear', align_corners=False)
        out = torch.sigmoid(out)  # 0~1 범위
        return out  # (B, 15, 128, 128)


# ===== Loss 정의: Huber + Pearson Correlation Loss =====
class HuberPearsonLoss(nn.Module):
    def __init__(self, huber_weight=1.0, pcc_weight=1.0, huber_delta=1.0):
        super().__init__()
        self.huber_weight = huber_weight
        self.pcc_weight = pcc_weight
        self.huber = nn.HuberLoss(delta=huber_delta)

    def pearson_loss(self, pred, target):
        B, C, H, W = pred.shape
        pred = pred.view(B, C, -1)
        target = target.view(B, C, -1)

        pred_c = pred - pred.mean(dim=2, keepdim=True)
        target_c = target - target.mean(dim=2, keepdim=True)

        cov = (pred_c * target_c).sum(dim=2)
        pred_std = pred_c.pow(2).sum(dim=2).sqrt()
        target_std = target_c.pow(2).sum(dim=2).sqrt()

        pcc = cov / (pred_std * target_std + 1e-8)
        return 1 - pcc.mean()

    def forward(self, pred, target):
        loss_huber = self.huber(pred, target)
        loss_pcc = self.pearson_loss(pred, target)
        return self.huber_weight * loss_huber + self.pcc_weight * loss_pcc


model = GeneExpressionModel(encoder_name='tu-convnext_tiny', num_genes=num_genes).to(device)

criterion = HuberPearsonLoss(huber_weight=1.0, pcc_weight=1.0)
optimizer = optim.Adam(model.parameters(), lr=lr)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# ===== Pearson Correlation Coefficient =====
def pearson_corr(pred, target):
    """gene별 PCC 계산 (batch 평균)"""
    B, C, H, W = pred.shape
    pred = pred.view(B, C, -1)    # (B, 15, H*W)
    target = target.view(B, C, -1)

    pred_mean = pred.mean(dim=2, keepdim=True)
    target_mean = target.mean(dim=2, keepdim=True)

    pred_centered = pred - pred_mean
    target_centered = target - target_mean

    cov = (pred_centered * target_centered).sum(dim=2)
    pred_std = pred_centered.pow(2).sum(dim=2).sqrt()
    target_std = target_centered.pow(2).sum(dim=2).sqrt()

    pcc = cov / (pred_std * target_std + 1e-8)  # (B, 15)
    return pcc.mean(dim=0)  # (15,) gene별 평균 PCC

# ===== 학습 루프 =====
train_loss_list = []
val_loss_list = []
train_pcc_list = []
val_pcc_list = []
MIN_loss = float('inf')

for epoch in range(num_epochs):
    # ===== Train =====
    model.train()
    train_bar = tqdm(train_dataloader, desc=f'Train {epoch+1}/{num_epochs}')
    running_loss = 0.0
    running_pcc = torch.zeros(num_genes)
    count = 0

    for x, y in train_bar:
        x = x.to(device).float()
        y = y.to(device).float()
        count += 1

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        with torch.no_grad():
            running_pcc += pearson_corr(pred, y).cpu()

        avg_pcc = running_pcc.mean().item() / count
        train_bar.set_postfix(loss=f'{running_loss/count:.4f}', pcc=f'{avg_pcc:.4f}')

    train_loss_list.append(running_loss / count)
    train_pcc_list.append((running_pcc / count).mean().item())

    # ===== Validation =====
    model.eval()
    val_bar = tqdm(val_dataloader, desc=f'Val   {epoch+1}/{num_epochs}')
    val_loss = 0.0
    val_pcc = torch.zeros(num_genes)
    count = 0

    with torch.no_grad():
        for x, y in val_bar:
            x = x.to(device).float()
            y = y.to(device).float()
            count += 1

            pred = model(x)
            loss = criterion(pred, y)

            val_loss += loss.item()
            val_pcc += pearson_corr(pred, y).cpu()

            avg_pcc = val_pcc.mean().item() / count
            val_bar.set_postfix(loss=f'{val_loss/count:.4f}', pcc=f'{avg_pcc:.4f}')

    val_loss_list.append(val_loss / count)
    val_pcc_list.append((val_pcc / count).mean().item())

    # ===== Best Model 저장 =====
    if MIN_loss > val_loss / count:
        torch.save(model.state_dict(), f'{model_path}train_20x_best.pt')
        MIN_loss = val_loss / count
        print(f'  -> Best model saved (val_loss: {MIN_loss:.6f})')

    # ===== 주기적 시각화 =====
    if epoch % 50 == 5:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].set_title('Loss')
        axes[0].plot(train_loss_list, label='train')
        axes[0].plot(val_loss_list, label='val')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('MSE Loss')
        axes[0].legend()

        axes[1].set_title('Pearson Correlation')
        axes[1].plot(train_pcc_list, label='train')
        axes[1].plot(val_pcc_list, label='val')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('PCC')
        axes[1].set_ylim([0, 1])
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# ===== 최종 그래프 =====
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].set_title('Loss')
axes[0].plot(train_loss_list, label='train')
axes[0].plot(val_loss_list, label='val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].set_title('Pearson Correlation')
axes[1].plot(train_pcc_list, label='train')
axes[1].plot(val_pcc_list, label='val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PCC')
axes[1].set_ylim([0, 1])
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nBest val loss: {MIN_loss:.6f}')
print(f'batch_size={batch_size}, img_size={img_size}, lr={lr}')

In [ ]:
# ===== 검증 시각화 =====
# Best 모델 로드
model.load_state_dict(torch.load(f'{model_path}train_20x_best.pt', map_location=device))
model.eval()

# 테스트 배치 추출
val_iter = iter(val_dataloader)
x_test, y_test = next(val_iter)
x_test = x_test.to(device).float()

with torch.no_grad():
    pred = model(x_test)

x_np = x_test.cpu().numpy()
y_np = y_test.cpu().numpy()
pred_np = pred.cpu().numpy()

# 샘플 2개, gene 5개 시각화
show_genes = [0, 4, 8, 11, 14]  # EPCAM, COL1A1, CD3D, MS4A1, PECAM1
n_samples = min(2, x_np.shape[0])

for sample_idx in range(n_samples):
    fig, axes = plt.subplots(2, len(show_genes) + 1, figsize=(3 * (len(show_genes) + 1), 6))

    # H&E 이미지
    img = np.transpose(x_np[sample_idx], (1, 2, 0))
    axes[0, 0].imshow(img)
    axes[0, 0].set_title('H&E Input')
    axes[0, 0].axis('off')
    axes[1, 0].imshow(img)
    axes[1, 0].set_title('H&E Input')
    axes[1, 0].axis('off')

    for i, gi in enumerate(show_genes):
        axes[0, i + 1].imshow(y_np[sample_idx, gi], cmap='hot', vmin=0, vmax=1)
        axes[0, i + 1].set_title(f'GT: {marker_genes[gi]}')
        axes[0, i + 1].axis('off')

        axes[1, i + 1].imshow(pred_np[sample_idx, gi], cmap='hot', vmin=0, vmax=1)
        axes[1, i + 1].set_title(f'Pred: {marker_genes[gi]}')
        axes[1, i + 1].axis('off')

    fig.suptitle(f'Sample {sample_idx + 1}: Ground Truth (top) vs Prediction (bottom)', fontsize=14)
    plt.tight_layout()
    plt.show()

# ===== Gene별 PCC/MAE 결과 =====
print('\n===== Gene-wise Evaluation on Validation Set =====')
all_pcc = torch.zeros(num_genes)
all_mae = torch.zeros(num_genes)
count = 0

with torch.no_grad():
    for x, y in tqdm(val_dataloader, desc='Evaluating'):
        x = x.to(device).float()
        y = y.to(device).float()
        pred = model(x)
        count += 1

        all_pcc += pearson_corr(pred, y).cpu()
        mae_per_gene = (pred - y).abs().mean(dim=(0, 2, 3)).cpu()  # (15,)
        all_mae += mae_per_gene

all_pcc /= count
all_mae /= count

print(f'{"Gene":>10s} | {"PCC":>8s} | {"MAE":>8s}')
print('-' * 32)
for i, gene in enumerate(marker_genes):
    print(f'{gene:>10s} | {all_pcc[i].item():8.4f} | {all_mae[i].item():8.4f}')
print('-' * 32)
print(f'{"Mean":>10s} | {all_pcc.mean().item():8.4f} | {all_mae.mean().item():8.4f}')